# F01 · 금융시계열의 구조와 정상성

**연구질문:** ETF 가격과 로그수익률의 정상성은 다른가? 수익률의 제곱에서 변동성 군집이 관찰되는가?

**데이터:** Yahoo Finance SPY (온라인 시도) 또는 교육용 합성 가격 시계열 (자동 대체). 생성된 합성 데이터는 금융시장 실증결과가 아닙니다.

**학습 흐름:** 개념 → 수식 → 자료 → 진단 → 그래프 → 해석 → 검증.


## 1. 핵심 이론과 수식\n\n로그수익률: $r_t=\ln(P_t)-\ln(P_{t-1})$.\n\n약정상성: $E(r_t)=\mu$, $\mathrm{Var}(r_t)=\sigma^2$, $\mathrm{Cov}(r_t,r_{t-k})=\gamma(k)$.\n\nADF의 귀무가설은 **단위근 존재**이고 KPSS의 귀무가설은 **정상성**이다. 단일 $p$값으로 정상성을 확정하지 않는다.\n

In [ ]:
%pip -q install yfinance statsmodels pandas numpy matplotlib

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import yfinance as yf
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox
SEED=42
print("Libraries ready")

In [ ]:
# 실제 데이터가 필요하면 True. 네트워크가 불안정하면 자동 합성자료 대체.
USE_LIVE_DATA = True
DATA_KIND = "synthetic"
try:
    if not USE_LIVE_DATA:
        raise RuntimeError("Offline sample requested")
    raw = yf.download("SPY", start="2015-01-01", end="2026-01-01", auto_adjust=True, progress=False)
    if raw.empty:
        raise RuntimeError("Empty remote response")
    if isinstance(raw.columns, pd.MultiIndex):
        # yfinance returns a MultiIndex in many versions
        price = raw[("Close", "SPY")].dropna().astype(float)
    else:
        price = raw["Close"].dropna().astype(float)
    DATA_KIND = "SPY (Yahoo Finance; online)"
except Exception as exc:
    print("Public data unavailable:", type(exc).__name__, str(exc)[:160])
    # fixed-seed locally generated synthetic random walk, independent of GitHub paths
    rng = np.random.default_rng(SEED)
    date = pd.bdate_range("2024-01-01", periods=520)
    price = pd.Series(100*np.exp(np.cumsum(rng.normal(0,0.011,len(date)))), index=date, name="Close")
    DATA_KIND = "synthetic random walk (NOT real ETF data)"
assert len(price)>50
price.index = pd.to_datetime(price.index).tz_localize(None)
print("Data kind:", DATA_KIND, "rows:", len(price), "range:", price.index.min().date(), price.index.max().date())

In [ ]:
returns = np.log(price).diff().dropna()
fig, axes = plt.subplots(2, 1, figsize=(11, 6))
price.plot(ax=axes[0], title=f"Price | {DATA_KIND}")
returns.plot(ax=axes[1], title="Daily log returns")
plt.tight_layout(); plt.show()
print(returns.describe().to_string())

## 2. ADF·KPSS 검정

**해석 가이드:** ADF p값이 작다면 단위근 귀무가설에 반하는 증거. KPSS p값이 작다면 정상성 귀무가설에 반하는 증거. 구조변화·추세 및 검정력 한계를 함께 논의한다.

In [ ]:
def diagnostics(series, label):
    x = series.dropna().to_numpy(dtype=float)
    adf_p = adfuller(x, autolag="AIC")[1]
    kp_p = kpss(x, regression="c", nlags="auto")[1]
    return {"series":label,"ADF_p":adf_p,"KPSS_p":kp_p}
results = pd.DataFrame([diagnostics(price,"Price"), diagnostics(returns,"Log return")])
print(results.round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(2,2,figsize=(11,6))
plot_acf(returns, lags=30, ax=ax[0,0], title="Returns ACF")
plot_pacf(returns, lags=30, ax=ax[0,1], title="Returns PACF", method="ywm")
plot_acf(returns**2, lags=30, ax=ax[1,0], title="Squared returns ACF")
plot_pacf(returns**2, lags=30, ax=ax[1,1], title="Squared returns PACF", method="ywm")
plt.tight_layout(); plt.show()
print("Returns Ljung-Box:", acorr_ljungbox(returns,lags=[10],return_df=True)["lb_pvalue"].iloc[0])
print("ARCH-LM p:", het_arch(returns*100,nlags=10)[1])

## 3. 결과 해석 작성

1. `DATA_KIND`가 **synthetic**인지 **실제 SPY**인지 반드시 먼저 표기한다.
2. 가격과 수익률의 ADF/KPSS 결과가 서로 어떻게 다른지 설명한다.
3. 제곱수익률 ACF와 ARCH-LM이 변동성 군집을 지지하는지 논의한다.
4. 검정 결과가 서로 모순되면 왜 그런지 구조변화·표본기간·추세 측면에서 토론한다.

## 4. 강건성 및 심화 연구과제
- 최근 1년 vs 전체기간 비교 (실제 데이터일 때)
- SPY 대신 QQQ 또는 KOSPI를 사용해 변화 비교
- 수익률 계산을 단순수익률로 바꾸어 민감도 확인

**제출:** 그래프 2개 이상, 검정표, 해석 단락, 한계 및 데이터 출처.
